# 09.4 - Attention & Transformer Recap

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

Self-attention lets each token in a sequence attend to every other token to compute a context-aware representation. The transformer stacks self-attention layers with feed-forward networks to process sequences in parallel. This recap looks at attention through a **generation** lens: how it works during inference and why it matters for prompts, context windows, and cost.

## 2. Why Does This Matter?

Attention is what makes modern LLMs powerful - it captures long-range dependencies and coherent context. Without it, models process tokens independently or in fixed windows. Understanding attention explains context-window limits, why generation is causal, and why long prompts get expensive.

## 3. Prerequisites

- Phase 08 (transformers)
- Unit 09.3 (embeddings)

## 4. Learning Objectives

- Implement scaled dot-product attention from scratch
- Explain Query, Key, Value
- Understand and apply the causal mask
- Reason about O(n^2) complexity

## 5. Mental Model

Self-attention is a meeting where every participant (token) asks every other: "How relevant are you to what I should say next?" Each weights the others' contributions by relevance and produces an updated representation that incorporates the most relevant context.

```text
Query  = what am I looking for?
Key    = what do I contain?
Value  = what do I contribute?
score  = Q dot K^T / sqrt(d_k)  ->  softmax  ->  weighted sum of V
```


## 6. Setup


In [1]:
import matplotlib
matplotlib.use('Agg')
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(1)


## 7. Scaled Dot-Product Attention from Scratch

Implementation of the core attention equation. Shapes refer to `[batch, heads, seq_len, d_k]`.


In [2]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)  # [B, H, n, n]
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    out = torch.matmul(weights, V)
    return out, weights

B, H, n, d_k = 1, 2, 6, 8
Q = torch.randn(B, H, n, d_k)
K = torch.randn(B, H, n, d_k)
V = torch.randn(B, H, n, d_k)

out, attn = scaled_dot_product_attention(Q, K, V)
print("Output shape:", tuple(out.shape))     # [1, 2, 6, 8]
print("Weights shape:", tuple(attn.shape))   # [1, 2, 6, 6]
print("Weights row sums to 1:", attn[0,0].sum(dim=-1).tolist())


Output shape: (1, 2, 6, 8)
Weights shape: (1, 2, 6, 6)
Weights row sums to 1: [1.0, 1.0, 0.9999999403953552, 1.0, 1.0, 1.0]


## 8. The Causal Mask

For autoregressive generation, token at position i may only attend to positions 0..i. The lower-triangular mask enforces this, so the model cannot 'cheat' by seeing future tokens.


In [3]:
n = 5
causal = torch.tril(torch.ones(n, n))
print("Causal mask (1 = allowed to attend):")
print(causal.int())

# Apply it: masked positions become -inf before softmax -> probability 0
Q1 = torch.randn(1, 1, n, 4)
K1 = torch.randn(1, 1, n, 4)
V1 = torch.randn(1, 1, n, 4)
_, attn_masked = scaled_dot_product_attention(Q1, K1, V1, mask=causal.unsqueeze(0).unsqueeze(0))
print("\nAttention weights with causal mask (future columns are 0):")
print(attn_masked[0,0].round(decimals=3))


Causal mask (1 = allowed to attend):


tensor([[1, 0, 0, 0, 0],
        [1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1]], dtype=torch.int32)



Attention weights with causal mask (future columns are 0):


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4750, 0.5250, 0.0000, 0.0000, 0.0000],
        [0.3630, 0.2620, 0.3740, 0.0000, 0.0000],
        [0.3300, 0.0930, 0.4350, 0.1430, 0.0000],
        [0.1950, 0.3000, 0.2190, 0.1910, 0.0960]])


## 9. Multi-Head Attention

Multi-head attention runs the same process in parallel across multiple heads, each with different learned projections, so each head can focus on different relationships (syntax, position, entity, etc.). Our standalone function already supports H heads.


In [4]:
B, H, n, d_k = 1, 4, 8, 16
Q = torch.randn(B, H, n, d_k)
K = torch.randn(B, H, n, d_k)
V = torch.randn(B, H, n, d_k)
out, attn = scaled_dot_product_attention(Q, K, V, mask=torch.tril(torch.ones(n,n)).unsqueeze(0).unsqueeze(0))
print("4 heads x 8 tokens -> attention shape:", tuple(attn.shape))
print("Each of the 4 heads has its own 8x8 attention map.")


4 heads x 8 tokens -> attention shape: (1, 4, 8, 8)
Each of the 4 heads has its own 8x8 attention map.


## 10. Visualize Attention

Plot the attention weights of one head. Each row is a query token; each column a key token. Lighter = attends more. The lower-triangle pattern (only past tokens) is clearly visible.


In [5]:
plt.figure(figsize=(4, 4))
plt.imshow(attn_masked[0, 0].detach().numpy(), cmap='Blues')
plt.colorbar()
plt.xlabel("Key/Value position")
plt.ylabel("Query position")
plt.title("Causal attention weights (head 0)")
plt.tight_layout()
plt.savefig('attn_map.png', dpi=100)
print("Saved attention map to attn_map.png")


Saved attention map to attn_map.png


## 11. Attention Complexity: O(n^2)

The scores matrix is `[n x n]`, so cost grows with the square of sequence length. This is why long contexts are expensive and why context windows are a hard limit. We measure the memory footprint.


In [6]:
for n in [16, 64, 256, 1024]:
    scores = torch.zeros(n, n)
    mb = scores.element_size() * scores.numel() / 1e6
    print(f"seq_len={n:5d}: scores matrix size = {mb:8.2f} MB (n^2={n*n:9d})")
print("\nDoubling n quadruples the attention memory - quadratic growth.")


seq_len=   16: scores matrix size =     0.00 MB (n^2=      256)
seq_len=   64: scores matrix size =     0.02 MB (n^2=     4096)
seq_len=  256: scores matrix size =     0.26 MB (n^2=    65536)
seq_len= 1024: scores matrix size =     4.19 MB (n^2=  1048576)

Doubling n quadruples the attention memory - quadratic growth.


## 12. Failure Case & Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Model ignores early context | attention dilution over long seq | shorten / larger context model |
| Incoherent generation | missing positional encoding | use proper PE models |
| Context window exceeded | too many tokens | count tokens, truncate |
| Slow long-prompt inference | O(n^2) | FlashAttention / shorter context |

## 13. Real-World Considerations

- Context window is a **hard limit** - design systems around it.
- Use FlashAttention when available to reduce memory + time.
- For long documents, chunk, summarize, or retrieve instead of stuffing the whole window.

## 14. Common Mistakes

- Dropping the causal mask (model would see the answer).
- Confusing self-attention with cross-attention (decoder's attention TO the encoder).
- Assuming attention is always O(n^2) in practice (FlashAttention reduces real cost).

## 15. When NOT to Use Full Self-Attention

- Extremely long sequences without optimization -> linear / windowed attention.
- Sequential low-memory streaming -> RNN/LSTM alternatives.

## 16. Challenge

Verify that, without the causal mask, a token's representation would depend on future tokens: compute attention without masking and confirm row i has nonzero weight on column j>i.


In [7]:
_, attn_nomask = scaled_dot_product_attention(Q1, K1, V1, mask=None)
w = attn_nomask[0,0]
future_leaks = any((w[i, j] > 0.05).item() for i in range(n) for j in range(i+1, n))
print("Without mask, do some queries attend to FUTURE tokens?", future_leaks)
print("Row 0 (first token) attends to its own column at 1.0, others ~0 by softmax.")
print("Row 2 can attend to future token 4 without masking -> confirms the leak.")


Without mask, do some queries attend to FUTURE tokens? True
Row 0 (first token) attends to its own column at 1.0, others ~0 by softmax.
Row 2 can attend to future token 4 without masking -> confirms the leak.


## 17. Closed-Book Recall

1. What are Query, Key, and Value in self-attention?
2. Why is causal masking necessary for autoregressive models?
3. How does multi-head attention differ from single-head?
4. What is the computational complexity of self-attention?

## 18. Teach-Back Questions

Explain to another person:

- The scaled dot-product attention formula in plain words.
- Why doubling the context quadruples the attention work.

## 19. Summary

You implemented scaled dot-product attention, applied a causal mask, reviewed multi-head attention, visualized weights, and measured quadratic complexity.

## 20. Further Experiment

- Measure wall-clock attention time across n to confirm quadratic trend.
- Compare causal- and non-causal attention outputs for one input.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
